In [ ]:
# Cell 1 — Install
!pip install -q ultralytics

In [ ]:
# Cell 2 — Config (change DATASET_NAME to your Kaggle dataset slug)
DATASET_NAME = "lesion-yolo-seg"  # ← change this

import yaml
from pathlib import Path
data_yaml = Path(f"/kaggle/input/datasets/<kaggle-user>/{DATASET_NAME}/data.yaml")
with open(data_yaml) as f:
    cfg = yaml.safe_load(f)

cfg["path"]  = str(data_yaml.parent)
cfg["train"] = "images/train"
cfg["val"]   = "images/val"

fixed_yaml = Path("/kaggle/working/data.yaml")
with open(fixed_yaml, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(fixed_yaml.read_text())

In [ ]:
# Cell 3 — Reduce background images to 20% of lesion count
import shutil
import yaml
from pathlib import Path

with open("/kaggle/working/data.yaml") as f:
    cfg = yaml.safe_load(f)

train_img_dir = Path(cfg["path"]) / cfg["train"]
train_lbl_dir = Path(str(train_img_dir).replace("images", "labels"))

all_imgs = sorted(p for p in train_img_dir.iterdir()
                  if p.suffix.lower() in {".jpg", ".jpeg", ".png"})

lesion_imgs = [p for p in all_imgs
               if (train_lbl_dir / f"{p.stem}.txt").exists()
               and (train_lbl_dir / f"{p.stem}.txt").stat().st_size > 0]
bg_imgs     = [p for p in all_imgs if p not in lesion_imgs]

max_bg    = int(len(lesion_imgs) * 0.2)
remove_bg = bg_imgs[max_bg:]

print(f"Lesion images : {len(lesion_imgs)}")
print(f"Background    : {len(bg_imgs)} total → keeping {max_bg}, removing {len(remove_bg)}")

# copy dataset to working dir (input is read-only)
WORK_DATA = Path("/kaggle/working/dataset_filtered")
if WORK_DATA.exists():
    shutil.rmtree(WORK_DATA)
shutil.copytree(Path(cfg["path"]), WORK_DATA)

for p in remove_bg:
    (WORK_DATA / "images" / "train" / p.name).unlink(missing_ok=True)
    (WORK_DATA / "labels" / "train" / f"{p.stem}.txt").unlink(missing_ok=True)

filtered_yaml = Path("/kaggle/working/data_filtered.yaml")
cfg["path"] = str(WORK_DATA)
with open(filtered_yaml, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f"\nFiltered dataset → {WORK_DATA}")
print(filtered_yaml.read_text())

In [ ]:
# Cell 4 — Train YOLOv8s-seg
from ultralytics import YOLO

model = YOLO("yolov8s-seg.pt")   # small > nano — better recall on small dataset

results = model.train(
    data          = "/kaggle/working/data_filtered.yaml",
    epochs        = 120,
    imgsz         = 640,
    batch         = 16,
    patience      = 25,
    optimizer     = "AdamW",
    lr0           = 5e-4,
    lrf           = 0.01,
    weight_decay  = 5e-4,
    warmup_epochs = 5,
    # stronger augmentation to boost recall
    hsv_h=0.02,  hsv_s=0.6,  hsv_v=0.4,
    degrees=20,  translate=0.15, scale=0.4,
    flipud=0.3,  fliplr=0.5,
    mosaic=1.0,
    copy_paste=0.5,
    project   = "/kaggle/working/yolo_runs",
    name      = "lesion_seg_v2",
    exist_ok  = True,
    device    = 0,
    workers   = 2,
)

BEST_PT = str(results.save_dir) + "/weights/best.pt"
print("Best weights:", BEST_PT)

In [ ]:
# Cell 5 — Quantitative validation
from ultralytics import YOLO

BEST_PT = "/kaggle/working/yolo_runs/lesion_seg_v2/weights/best.pt"
model   = YOLO(BEST_PT)
metrics = model.val(data="/kaggle/working/data_filtered.yaml",
                    imgsz=640, device=0, verbose=True)

print("\n=== Validation Metrics ===")
print(f"  mAP50      (box) : {metrics.box.map50:.4f}")
print(f"  mAP50-95   (box) : {metrics.box.map:.4f}")
print(f"  mAP50      (seg) : {metrics.seg.map50:.4f}")
print(f"  mAP50-95   (seg) : {metrics.seg.map:.4f}")
print(f"  Precision        : {metrics.box.mp:.4f}")
print(f"  Recall           : {metrics.box.mr:.4f}")

In [ ]:
# Cell 6 — Training curves
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("/kaggle/working/yolo_runs/lesion_seg_v2/results.csv")
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("YOLOv8s-seg Training", fontsize=12)

axes[0].plot(df["epoch"], df["train/seg_loss"], label="train")
axes[0].plot(df["epoch"], df["val/seg_loss"],   label="val", linestyle="--")
axes[0].set_title("Seg Loss"); axes[0].legend(); axes[0].grid(True)

axes[1].plot(df["epoch"], df["metrics/mAP50(M)"],    label="mAP50")
axes[1].plot(df["epoch"], df["metrics/mAP50-95(M)"], label="mAP50-95", linestyle="--")
axes[1].set_title("Seg mAP"); axes[1].legend(); axes[1].grid(True)

axes[2].plot(df["epoch"], df["metrics/precision(B)"], label="precision")
axes[2].plot(df["epoch"], df["metrics/recall(B)"],    label="recall")
axes[2].set_title("Precision / Recall"); axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.savefig("/kaggle/working/training_curves.png", dpi=120)
plt.show()

In [ ]:
# Cell 7 — Overlay val images: original | GT (blue) | prediction (green)
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

BEST_PT  = "/kaggle/working/yolo_runs/lesion_seg_v2/weights/best.pt"
VAL_DIR  = Path("/kaggle/working/dataset_filtered/images/val")
LBL_DIR  = Path("/kaggle/working/dataset_filtered/labels/val")
OUT_DIR  = Path("/kaggle/working/val_overlays")
OUT_DIR.mkdir(exist_ok=True)

N_SHOW      = 16
CONF        = 0.15    # low conf — YOLO is a gate, missing lesions is worse than false alarms
PRED_COLOR  = (50, 220,  50)
GT_COLOR    = (50,  50, 220)
ALPHA       = 0.40
EXTS        = {".jpg", ".jpeg", ".png"}

model = YOLO(BEST_PT)


def draw_gt(img, lbl_path):
    out = img.copy()
    if not lbl_path.exists() or lbl_path.stat().st_size == 0:
        return out
    H, W = img.shape[:2]
    ov = out.copy()
    for line in lbl_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) < 7:
            continue
        coords = list(map(float, parts[1:]))
        pts = np.array([(int(coords[i]*W), int(coords[i+1]*H))
                        for i in range(0, len(coords), 2)], dtype=np.int32)
        cv2.fillPoly(ov, [pts], GT_COLOR)
        cv2.polylines(out, [pts], True, GT_COLOR, 2)
    cv2.addWeighted(ov, ALPHA, out, 1-ALPHA, 0, out)
    return out


def draw_pred(img, result):
    out = img.copy()
    if result.masks is None:
        return out
    H, W = img.shape[:2]
    ov = out.copy()
    for mask_t, conf in zip(result.masks.data, result.boxes.conf):
        mask   = cv2.resize(mask_t.cpu().numpy(), (W, H), interpolation=cv2.INTER_LINEAR)
        binary = (mask > 0.5).astype(np.uint8)
        ov[binary == 1] = PRED_COLOR
        cnts, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(out, cnts, -1, PRED_COLOR, 2)
        if cnts:
            x, y, w, h = cv2.boundingRect(cnts[0])
            cv2.putText(out, f"{conf:.2f}", (x, max(y-4, 12)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, PRED_COLOR, 1)
    cv2.addWeighted(ov, ALPHA, out, 1-ALPHA, 0, out)
    return out


def add_header(img, labels, panel_w):
    bar = np.zeros((22, img.shape[1], 3), dtype=np.uint8)
    for i, t in enumerate(labels):
        cv2.putText(bar, t, (i*panel_w+6, 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)
    return np.concatenate([bar, img], axis=0)


# prioritise lesion images in display
all_imgs    = sorted(p for p in VAL_DIR.iterdir() if p.suffix.lower() in EXTS)
lesion_imgs = [p for p in all_imgs
               if (LBL_DIR / f"{p.stem}.txt").exists()
               and (LBL_DIR / f"{p.stem}.txt").stat().st_size > 0]
bg_imgs     = [p for p in all_imgs if p not in set(lesion_imgs)]
show_imgs   = (lesion_imgs + bg_imgs)[:N_SHOW]

print(f"Showing {len(show_imgs)} images (lesion first) ...")

saved = []
for img_path in show_imgs:
    img_bgr  = cv2.imread(str(img_path))
    img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    result   = model(img_bgr, conf=CONF, verbose=False)[0]
    lbl_path = LBL_DIR / f"{img_path.stem}.txt"
    has_gt   = lbl_path.exists() and lbl_path.stat().st_size > 0
    has_pred = result.masks is not None

    row = np.concatenate([img_rgb, draw_gt(img_rgb, lbl_path), draw_pred(img_rgb, result)], axis=1)
    row = add_header(row, ["original", "GT (blue)", f"pred conf>{CONF} (green)"], img_rgb.shape[1])

    # title: show TP/FP/FN at image level
    status = "TP" if has_gt and has_pred else \
             "FN" if has_gt and not has_pred else \
             "FP" if not has_gt and has_pred else "TN"

    cv2.imwrite(str(OUT_DIR / f"{status}_{img_path.stem}.jpg"),
                cv2.cvtColor(row, cv2.COLOR_RGB2BGR))
    saved.append((row, f"[{status}] {img_path.stem}"))

# display grid
cols = 2
rows = (len(saved) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(22, 5*rows))
axes = np.array(axes).flatten()
for ax, (img, title) in zip(axes, saved):
    ax.imshow(img); ax.set_title(title, fontsize=8); ax.axis("off")
for ax in axes[len(saved):]:
    ax.axis("off")
plt.suptitle(f"Val overlays  GT(blue) | Pred(green)  conf>{CONF}  [TP/FP/FN/TN]",
             fontsize=11)
plt.tight_layout()
plt.savefig("/kaggle/working/val_grid.png", dpi=100)
plt.show()

# summary counts
from collections import Counter
counts = Counter(t.split(']')[0][1:] for _, t in saved)
print("\nImage-level counts:", dict(counts))
print(f"Saved overlays → {OUT_DIR}")

In [ ]:
# Cell 8 — Save best weights to output tab
import shutil
shutil.copy("/kaggle/working/yolo_runs/lesion_seg_v2/weights/best.pt",
            "/kaggle/working/yolo_best.pt")
print("Done — download yolo_best.pt from the output tab")

In [ ]:
import os
from IPython.display import FileLink

# Replace 'your_folder_name' with the actual folder you want to zip
os.chdir(r'/kaggle/working')
!zip -r val_overlays.zip val_overlays